In [166]:
import pandas as pd
import json
from pathlib import Path
from elasticsearch import Elasticsearch, helpers
from pprint import pprint

In [167]:
# Dyanmic Paths
BASE_DIR = Path.cwd()                        
DATA_DIR = BASE_DIR / "IR2025"    #removed /ir2025
CSV_PATH = DATA_DIR / "documents.csv"
JSONL_PATH = DATA_DIR / "documents.jsonl"
QUERIES_PATH = DATA_DIR / "queries.csv"
QRELS_PATH = DATA_DIR / "qrels.csv"

print(f"Data folder: {DATA_DIR}")
print(f"Input file:  {CSV_PATH.name}")
print(f"Output file: {JSONL_PATH.name}")

Data folder: c:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\IR2025
Input file:  documents.csv
Output file: documents.jsonl


In [168]:
# Load CSV File
try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8")
except FileNotFoundError:
    raise SystemExit(f"File not found: {CSV_PATH}")
except Exception as e:
    raise SystemExit(f"Error reading CSV: {e}")

In [169]:
# Display Columns and Document Count
print(f"CSV loaded successfully → {len(df)} documents found")
print(f"Columns detected: {list(df.columns)}")

CSV loaded successfully → 18316 documents found
Columns detected: ['ID', 'Text']


In [171]:
# Validate Required Columns
required_cols = {"ID", "Text"}
missing = required_cols - set(df.columns)
if missing:
    raise SystemExit(f"Missing expected columns: {missing}")

In [172]:
# Convert to JSONL
records_written = 0
with open(JSONL_PATH, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        record = {
            "id": row["ID"],     
            "text": row["Text"]
        }
        json_line = json.dumps(record, ensure_ascii=False)
        f.write(json_line + "\n")
        records_written += 1

print(f"Converted {records_written} rows → JSONL format")
print(f"Output saved at: {JSONL_PATH.resolve()}")


#edo den katalaino anoigeis jsonl gia an to kaneis jsonl? Mallon prepei na anoigei to csv prota

Converted 18316 rows → JSONL format
Output saved at: C:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\IR2025\documents.jsonl


In [181]:
class Search:
    def __init__(self):
        self.es = Elasticsearch("http://127.0.0.1:9200")
        client_info = self.es.info()
        print('Connected to Elasticsearch!')
        pprint(client_info.body)
        
    def get_es(self):
        return self.es
        
    def create_index(self):
        self.es.indices.delete(index='ir2025', ignore_unavailable=True)
        index_settings = {
            "settings": {
                "similarity": {
                    "default": {"type": "BM25"}
                },
                "analysis": {
                    "analyzer": {
                        "default": {
                            "type": "standard"
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "id": {"type": "keyword"},
                    "text": {"type": "text", "analyzer": "standard"}
                }
            }
        }

        self.es.indices.create(index='ir2025', body=index_settings)
        print("Index created with English analyzer and BM25 similarity.")

        
    def insert_document(self, document):
        return self.es.index(index='ir2025', document=document)
    
    def insert_documents(self, documents):
        operations = []
        for document in documents:
            operations.append({'index': {'_index': 'ir2025'}})
            operations.append(document)

        result = self.es.bulk(operations=operations)
        print("Insertion finished.")
        print(result) 
    
    def delete_index(self, index_name="ir2025"):
        if self.es.indices.exists(index=index_name):
            self.es.indices.delete(index=index_name)
            print(f"Index '{index_name}' deleted successfully.")
        else:
            print(f"Index '{index_name}' does not exist.")

    def exists(self, index_name="ir2025"):
            try:
                return self.es.indices.exists(index=index_name)
            except Exception as e:
                print(f"Error checking if index exists: {e}")
                return False
            
    def generate_actions(self,jsonl_path, index_name):
        with open(jsonl_path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                yield {
                    "_index": index_name,
                    "_id": i,  
                    "_source": json.loads(line)
                }
                
    def index_documents(self,jsonl_path, index_name):
        actions = self.generate_actions(jsonl_path, index_name)
        success, _ = helpers.bulk(self.get_es(), actions)
        print(f"Successfully indexed {success} documents into '{index_name}'")
        
    def search_query(self, query_text, k=10, index_name="ir2025"):
        try:
            resp = self.es.search(
                index=index_name,
                query={"match": {"text": {"query": query_text}}},
                size=k
            )
            return resp["hits"]["hits"]
        except Exception as e:
            print(f"Error executing search query: {e}")
            return []
        
    def load_file(self,filename):
        df = pd.read_csv(filename, encoding="utf-8")
        print("Loaded queries successfully")
        return df
        
    def generate_file_results(self,dir,filename):
        df_queries=self.load_file(filename)
        for k in [20, 30, 50]:
            results_path = dir / f"results_{k}.txt"
            with open(results_path, "w", encoding="utf-8") as f:
                for _, row in df_queries.iterrows():
                    qid = str(row[0])
                    qtext = str(row[1])
                    results = self.search_query(qtext, k)
                    for rank, hit in enumerate(results, start=1):
                        docid = hit["_source"]["id"]
                        score = hit["_score"]
                        f.write(f"{qid} Q0 {docid} {rank} {score:.4f} BM25\n")
            print(f"Created results file: {results_path}")

    def load_qrels(self, path):
        return pd.read_csv(
            path,
            sep=";",             # <-- correct delimiter
            header=0,            # skip the "Column1;Column2;..." header
            encoding="utf-8-sig",# handles the BOM \ufeff
            names=["qid", "_", "docid", "rel"]
        )
    
    def load_results(self,path):
        return pd.read_csv(path, sep=r"\s+", header=None, names=["qid", "_", "docid", "rank", "score", "method"])
    
    def precision_at_k(self,results, qrels, k):
        precisions = []
        for qid, group in results.groupby("qid"):
            rel_docs = set(qrels[qrels["qid"] == qid]["docid"])
            retrieved = group.sort_values("rank").head(k)
            hits = sum(doc in rel_docs for doc in retrieved["docid"])
            precisions.append(hits / k)
        return sum(precisions) / len(precisions)
    
    def mean_average_precision(self, results, qrels):
        APs = []
        for qid, group in results.groupby("qid"):
            rel_docs = set(qrels[qrels["qid"] == qid]["docid"])
            if not rel_docs:  # Skip queries with no relevance judgments
                continue
            retrieved = group.sort_values("rank")
            hits, cum_prec = 0, 0.0
            for i, doc in enumerate(retrieved["docid"], 1):
                if doc in rel_docs:
                    hits += 1
                    cum_prec += hits / i
            if hits > 0:
                APs.append(cum_prec / hits)
        return sum(APs) / len(APs) if APs else 0.0

    
    def evaluate_results(self,qrels):
        for k in [20, 30, 50]:
            results = self.load_results(DATA_DIR / f"results_{k}.txt")
            print(f"\nEvaluation for results_{k}.txt")
            for kk in [5, 10, 15, 20]:
                p = self.precision_at_k(results, qrels, kk)
                print(f"Precision@{kk}: {p:.4f}")
        map_val = self.mean_average_precision(results, qrels)
        print(f"MAP: {map_val:.4f}")

In [182]:
search=Search()
INDEX_NAME = "ir2025" 

Connected to Elasticsearch!
{'cluster_name': 'elasticsearch',
 'cluster_uuid': 'NfFciNgwSE648-kMzsJiVg',
 'name': 'SWKRATISLAPTOP',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2025-09-16T22:05:19.073893347Z',
             'build_flavor': 'default',
             'build_hash': '0b7fe68d2e369469ff9e9f344ab6df64ab9c5293',
             'build_snapshot': False,
             'build_type': 'zip',
             'lucene_version': '10.2.2',
             'minimum_index_compatibility_version': '8.0.0',
             'minimum_wire_compatibility_version': '8.19.0',
             'number': '9.1.4'}}


In [183]:
if search.exists(INDEX_NAME):
    search.delete_index(index_name=INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'")

search.create_index()

print(f"Created new index '{INDEX_NAME}' with BM25 similarity")


Index 'ir2025' deleted successfully.
Deleted existing index 'ir2025'
Index created with English analyzer and BM25 similarity.
Created new index 'ir2025' with BM25 similarity


In [184]:
#Check if index exists
res = search.exists(index_name=INDEX_NAME)
print(res)
#Index docs
search.index_documents(JSONL_PATH,INDEX_NAME)
#Generate results
search.generate_file_results(DATA_DIR,QUERIES_PATH)

True
Successfully indexed 18316 documents into 'ir2025'
Loaded queries successfully
Created results file: c:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\IR2025\results_20.txt


C:\Users\swkra\AppData\Local\Temp\ipykernel_16512\4093027370.py:102: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  qid = str(row[0])
C:\Users\swkra\AppData\Local\Temp\ipykernel_16512\4093027370.py:103: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  qtext = str(row[1])


Created results file: c:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\IR2025\results_30.txt
Created results file: c:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\IR2025\results_50.txt


In [ ]:
#load qrels
qrels = search.load_qrels(QRELS_PATH)
# Evaluate Results
search.evaluate_results(qrels=qrels)


Evaluation for results_20.txt
Precision@5: 0.6600
Precision@10: 0.5700
Precision@15: 0.4667
Precision@20: 0.4000

Evaluation for results_30.txt
Precision@5: 0.6800
Precision@10: 0.6000
Precision@15: 0.5000
Precision@20: 0.4250

Evaluation for results_50.txt
Precision@5: 0.6800
Precision@10: 0.6000
Precision@15: 0.5000
Precision@20: 0.4250
MAP: 0.6272


In [186]:
##### THA TO FTIKSW ME PINAKA KANONIKO KAI OXI AYTH THN AHDIA

# Visualization of Precision@k
import matplotlib.pyplot as plt
import seaborn as sns

# RESULTS from evaluation
data = {
    "Retrieval_k": [20, 30, 50],
    "P@5":  [0.82, 0.82, 0.82],
    "P@10": [0.71, 0.71, 0.71],
    "P@15": [0.5867, 0.5867, 0.5867],
    "P@20": [0.52, 0.52, 0.52],
    "MAP":  [0.8039, 0.7670, 0.7039]
}

df = pd.DataFrame(data)

sns.set(style="whitegrid", context="talk")
palette = sns.color_palette("crest", as_cmap=False)

fig, ax1 = plt.subplots(figsize=(10, 6))
precision_cols = ["P@5", "P@10", "P@15", "P@20"]
df_melted = df.melt(id_vars="Retrieval_k", value_vars=precision_cols,
                    var_name="Metric", value_name="Score")

sns.barplot(
    data=df_melted, x="Retrieval_k", y="Score", hue="Metric",
    palette=palette, ax=ax1
)

ax1.set_title("Precision@k for Different Retrieval Depths (BM25 Baseline)", fontsize=16, weight="bold")
ax1.set_xlabel("Number of Retrieved Documents (k)", fontsize=13)
ax1.set_ylabel("Precision", fontsize=13)
ax1.legend(title="Metric", loc="upper right")
ax1.set_ylim(0, 1)

for container in ax1.containers:
    ax1.bar_label(container, fmt="%.2f", label_type="edge", fontsize=10, padding=3)

plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'